In [1]:
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

from qiskit import transpile
from qiskit.circuit import Parameter,ParameterExpression
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeMumbaiV2
from qiskit.converters import circuit_to_dag, dag_to_circuit

import sys
sys.path.append("../")
from clapton.clapton import claptonize
from clapton.circuit_manipulation import transform_to_allowed_gates,qiskit_to_stim, modify_circuit, multi_angle_qaoa_circuit, transform_qiskit_to_stim,generate_qiskit_param_map
from testing_scripts.graphs_utils import generate_random_complete_graph,generate_k_regular_graph,compute_optimal_max_cut,build_max_cut_paulis
from testing_scripts.qaoa_utils import QAOASolver

In [2]:
n = 10
G=generate_random_complete_graph(num_vertices=n, weighted=True, seed=True)

In [3]:
max_cut_paulis = build_max_cut_paulis(G)

cost_hamiltonian = SparsePauliOp.from_list(max_cut_paulis)
print("Cost Function Hamiltonian:", cost_hamiltonian)

Cost Function Hamiltonian: SparsePauliOp(['IIIIIIIIZZ', 'IIIIIIIZIZ', 'IIIIIIZIIZ', 'IIIIIZIIIZ', 'IIIIZIIIIZ', 'IIIZIIIIIZ', 'IIZIIIIIIZ', 'IZIIIIIIIZ', 'ZIIIIIIIIZ', 'IIIIIIIZZI', 'IIIIIIZIZI', 'IIIIIZIIZI', 'IIIIZIIIZI', 'IIIZIIIIZI', 'IIZIIIIIZI', 'IZIIIIIIZI', 'ZIIIIIIIZI', 'IIIIIIZZII', 'IIIIIZIZII', 'IIIIZIIZII', 'IIIZIIIZII', 'IIZIIIIZII', 'IZIIIIIZII', 'ZIIIIIIZII', 'IIIIIZZIII', 'IIIIZIZIII', 'IIIZIIZIII', 'IIZIIIZIII', 'IZIIIIZIII', 'ZIIIIIZIII', 'IIIIZZIIII', 'IIIZIZIIII', 'IIZIIZIIII', 'IZIIIZIIII', 'ZIIIIZIIII', 'IIIZZIIIII', 'IIZIZIIIII', 'IZIIZIIIII', 'ZIIIZIIIII', 'IIZZIIIIII', 'IZIZIIIIII', 'ZIIZIIIIII', 'IZZIIIIIII', 'ZIZIIIIIII', 'ZZIIIIIIII'],
              coeffs=[ 7.+0.j,  7.+0.j,  1.+0.j,  5.+0.j,  9.+0.j,  8.+0.j,  7.+0.j,  5.+0.j,
  8.+0.j,  6.+0.j, 10.+0.j,  4.+0.j,  9.+0.j,  3.+0.j,  5.+0.j,  3.+0.j,
  2.+0.j, 10.+0.j,  5.+0.j,  9.+0.j, 10.+0.j,  3.+0.j,  5.+0.j,  2.+0.j,
  2.+0.j,  6.+0.j,  8.+0.j,  9.+0.j,  2.+0.j,  6.+0.j,  7.+0.j,  6.+0.j,
 10.+0.j,  4.+

In [4]:
reps = 2
circuit = multi_angle_qaoa_circuit(n,G ,reps)
circuit.num_parameters

110

In [5]:
maxcut_qaoa = QAOASolver(cost_hamiltonian,circuit)
maxcut_qaoa.prepare_circuit()

In [6]:
#Run CAFQA process 
maxcut_qaoa.run_CAFQA(n_gens=100)

STARTING ROUND 0


started GA at id 1 with 1 procs


started GA at id 2 with 1 procs
started GA at id 3 with 1 procs

started GA at id None with 1 procs

GA parameters used for this experiment:
  num_generations=50
  num_parents_mating=20
  population_size=100
  num_genes=110
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5
GA parameters used for this experiment:
  num_generations=50
  num_parents_mating=20
  population_size=100
  num_genes=110
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5
GA parameters used for this experiment:
  num_generations=50
  num_parents_mating=20
  population_size=100
  num_genes=110
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=

In [7]:
maxcut_qaoa.evaluate_exact_energy()

Exact Energy from Eigensolver: -83.0


np.float64(-83.0)

In [11]:
np.unique(maxcut_qaoa.best_cafqa_gen_fitness)

array([-33., -31., -26., -17., -11., -10.,  -8.,  -5.])

In [14]:
def select_alternate_elements(arr, num_elements=5):
    return arr[::2][:num_elements]

# Example usage
arr = [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
selected = select_alternate_elements(np.unique(maxcut_qaoa.best_cafqa_gen_fitness))
print(selected)

[-33. -26. -11.  -8.]


In [25]:
a=np.unique(maxcut_qaoa.best_cafqa_gen_fitness)[::3][:7]
a

array([-33., -17.,  -8.])

In [ ]:
list(a).index(-8)

2

In [29]:
indices = [0, 2, 4]  # Indices of elements to extract
elements = np.array(arr)[indices]  # Using numpy for indexing
print(elements)

[10  8  6]


In [34]:
for i,j in enumerate(zip(indices,elements)):
    print(i,j[0],j[1])

0 0 10
1 2 8
2 4 6
